# SQL 탐정단 1차시
## 뒤바뀐 무선이어폰 택배
**오늘의 임무:** 배송 기록에서 확인할 택배 후보 한 건을 찾고 근거를 설명합니다.

김부장: “이어폰을 주문했는데 다른 물건이 왔어. 보관함에 내 택배가 남아 있을까?”

혼이: “기록으로 후보부터 찾아봐요. 누가 잘못했는지는 아직 몰라요.”

**단서:** 물품은 무선이어폰 / 상태는 보관중 / 포장 무게 200~300g(양 끝 포함) / B구역

모든 자료는 수업용 가상 데이터입니다. 다른 사람의 실제 택배 정보는 조회하지 않습니다.

### 코랩 사용 순서
1. [Google Colab](https://colab.research.google.com/) → 파일 → 노트북 열기 → 업로드에서 이 ipynb를 엽니다.
2. 필요하면 Drive에 사본을 저장합니다. Python 3 런타임을 사용합니다. GPU는 필요 없습니다.
3. 준비 셀부터 아래 순서대로 실행합니다. 코드 셀 왼쪽 ▶ 또는 Shift+Enter를 누릅니다.
4. TODO 셀의 **삼중따옴표 안 SQL만** 바꿉니다. `run(...)` 등 Python 코드는 그대로 둡니다.
5. 마지막 채점 셀을 실행합니다. 고친 TODO 셀을 다시 실행해야 채점에 반영됩니다.
6. 파일 → 다운로드 → ipynb와 마지막 CSV를 제출합니다.

처음 배우는 친구는 1~6번을 먼저 완성하고, 예시를 참고해 7~10번에 도전하세요. 전체 권장 시간은 50분입니다.

## 0 준비
아래 셀은 가상 데이터베이스와 SQL 실행 도우미를 만듭니다. 설치나 DB 파일 업로드가 필요 없습니다. 다시 실행해도 답안 변수는 지우지 않습니다.

In [ ]:
import sqlite3, csv, io, json, re
from collections import Counter
from IPython.display import display, HTML
from html import escape

ROWS = [('P1001', '무선이어폰', 180, 'B', 'B01', '15:40', '보관중'), ('P1002', '휴대폰케이스', 90, 'B', 'B02', '16:10', '보관중'), ('P1003', '무선이어폰', 250, 'A', 'A02', '16:15', '수령완료'), ('P1004', '보조배터리', 280, 'B', 'B04', '16:20', '보관중'), ('P1005', '무선이어폰', 200, 'A', 'A03', '16:25', '보관중'), ('P1006', '무선이어폰', 310, 'B', 'B05', '16:30', '보관중'), ('P1007', '무선이어폰', 240, 'B', 'B03', '16:35', '보관중'), ('P1008', '무선이어폰', 300, 'B', 'B06', '16:40', '수령완료'), ('P1009', '무선이어폰', 300, 'A', 'A04', '16:45', '보관중'), ('P1010', '충전케이블', 70, 'C', 'C01', '16:50', '보관중'), ('P1011', '무선이어폰', 199, 'B', 'B07', '16:55', '보관중'), ('P1012', '무선이어폰', 301, 'B', 'B08', '17:00', '보관중'), ('P1013', '키보드', 750, 'C', 'C02', '17:05', '보관중'), ('P1014', '무선이어폰', 260, 'C', 'C03', '17:10', '보관중'), ('P1015', '노트', 220, 'B', 'B09', '17:15', '보관중'), ('P1016', '무선이어폰', 200, 'C', 'C04', '17:20', '수령완료')]
COLUMNS = ['parcel_id', 'item', 'weight_g', 'zone', 'locker', 'arrived_at', 'status']
SCHEMA = """CREATE TABLE parcels(
 parcel_id TEXT PRIMARY KEY, item TEXT, weight_g INTEGER,
 zone TEXT, locker TEXT, arrived_at TEXT, status TEXT);"""

def make_db(data=None):
    c = sqlite3.connect(":memory:")
    c.execute(SCHEMA)
    c.executemany("INSERT INTO parcels VALUES (?,?,?,?,?,?,?)", ROWS if data is None else data)
    c.commit()
    return c

def read_query(sql, data=None):
    if not isinstance(sql, str) or not sql.strip():
        raise ValueError("SQL을 입력해 주세요.")
    if not re.match(r"^\s*SELECT\b", sql, re.I):
        raise ValueError("SELECT로 시작하는 조회문 한 개를 입력하세요.")
    c = make_db(data)
    c.execute("PRAGMA query_only = ON")
    def authorizer(action, a, b, db, src):
        allowed = {sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ, sqlite3.SQLITE_FUNCTION}
        return sqlite3.SQLITE_OK if action in allowed else sqlite3.SQLITE_DENY
    c.set_authorizer(authorizer)
    ticks = [0]
    def progress():
        ticks[0] += 1
        return int(ticks[0] > 200)
    c.set_progress_handler(progress, 1000)
    try:
        cur = c.execute(sql)
        names = [d[0] for d in cur.description]
        result = cur.fetchmany(501)
        if len(result) > 500:
            raise ValueError("500행을 넘었습니다. 조회 조건을 확인하세요.")
        return names, result
    finally:
        c.close()

def run(sql):
    try:
        names, result = read_query(sql)
        header = ''.join('<th>'+escape(str(v))+'</th>' for v in names)
        body = ''.join('<tr>'+''.join('<td>'+escape(str(v))+'</td>' for v in row)+'</tr>' for row in result)
        display(HTML('<div style="overflow:auto"><table border="1" cellpadding="6"><tr>'+header+'</tr>'+body+'</table></div>'))
        print(f"조회 결과: {len(result)}행")
    except (sqlite3.Error, ValueError) as e:
        print("실행 안내:", e)

print("준비 완료: 가상 배송 기록 16건. 외부 파일이나 추가 설치가 필요 없습니다.")
run("SELECT * FROM parcels;")


## 표를 읽는 방법
`parcels`는 배송 기록을 담은 **테이블(표)** 입니다. **행** 하나는 택배 한 건, **열** 하나는 정보 한 종류입니다. **DB**는 이런 데이터를 모아 관리하는 곳이고, **SQL**은 DB에 원하는 데이터를 요청하는 언어입니다.

|열|뜻|예시|
|---|---|---|
|parcel_id|택배를 구분하는 고유 번호|P1001|
|item|물품 종류|무선이어폰|
|weight_g|포장 무게, 숫자 단위 g|180|
|zone|보관함 구역|B|
|locker|보관함 칸|B01|
|arrived_at|도착 시각, HH:MM 문자|15:40|
|status|처리 상태|보관중|

엑셀의 필터와 정렬을 떠올리면 쉽습니다. SQL은 같은 요청을 문장으로 저장하고 다시 실행할 수 있습니다.

## 함께 해 보는 예시
`SELECT` 뒤에는 볼 열, `FROM` 뒤에는 표, `WHERE` 뒤에는 조건을 씁니다.

```sql
SELECT parcel_id, item
FROM parcels
WHERE zone = 'C';
```

위 코드는 C구역 택배의 번호와 물품만 보여 줍니다. 글자는 `'작은따옴표'`, 숫자는 `200`처럼 씁니다.

- `*`: 모든 열
- `>=`: 이상(같은 값 포함), `<=`: 이하(같은 값 포함)
- `AND`: 앞뒤 조건을 모두 만족
- `ORDER BY weight_g ASC, parcel_id ASC`: 무게 오름차순, 동점은 번호 오름차순

`WHERE weight_g >= 100 AND weight_g <= 800`처럼 범위를 묶을 수 있습니다.
`ORDER BY`가 없으면 행 순서는 보장되지 않습니다. 본 과제는 9번만 행 순서까지 채점합니다.

In [ ]:
run("SELECT parcel_id, item FROM parcels WHERE zone = 'C';")

## TODO 01 배송 기록 전체 열기
**미션:** 모든 열을 조회하세요. 행 순서는 자유입니다.

<details><summary>막힐 때 힌트</summary>

*는 모든 열을 뜻합니다.

</details>

In [ ]:
# TODO 01: 삼중따옴표 안을 완성하세요.
sql_01 = """
SELECT ___ FROM parcels;
"""
run(sql_01)

## TODO 02 필요한 정보만 선택
**미션:** parcel_id, item, locker 열을 이 순서로 조회하세요.

<details><summary>막힐 때 힌트</summary>

열과 열 사이에는 쉼표를 씁니다.

</details>

In [ ]:
# TODO 02: 삼중따옴표 안을 완성하세요.
sql_02 = """
SELECT ___, ___, ___ FROM parcels;
"""
run(sql_02)

## TODO 03 무선이어폰만 찾기
**미션:** item이 '무선이어폰'인 행의 모든 열을 조회하세요.

<details><summary>막힐 때 힌트</summary>

글자는 작은따옴표로 감쌉니다.

</details>

In [ ]:
# TODO 03: 삼중따옴표 안을 완성하세요.
sql_03 = """
SELECT * FROM parcels WHERE ___ = '___';
"""
run(sql_03)

## TODO 04 수령이 끝난 택배
**미션:** status가 '수령완료'인 행의 모든 열을 조회하세요.

<details><summary>막힐 때 힌트</summary>

보관중과 수령완료는 다른 상태입니다.

</details>

In [ ]:
# TODO 04: 삼중따옴표 안을 완성하세요.
sql_04 = """
SELECT * FROM parcels WHERE ___ = '___';
"""
run(sql_04)

## TODO 05 200g 이상 찾기
**미션:** weight_g가 200 이상인 행의 모든 열을 조회하세요. 200도 포함합니다.

<details><summary>막힐 때 힌트</summary>

숫자에는 g와 따옴표를 붙이지 않습니다.

</details>

In [ ]:
# TODO 05: 삼중따옴표 안을 완성하세요.
sql_05 = """
SELECT * FROM parcels WHERE ___ >= ___;
"""
run(sql_05)

## TODO 06 300g 이하 찾기
**미션:** weight_g가 300 이하인 행의 모든 열을 조회하세요. 300도 포함합니다.

<details><summary>막힐 때 힌트</summary>

<= 는 작거나 같다는 뜻입니다.

</details>

In [ ]:
# TODO 06: 삼중따옴표 안을 완성하세요.
sql_06 = """
SELECT * FROM parcels WHERE ___ <= ___;
"""
run(sql_06)

## TODO 07 아직 남아 있는 이어폰
**미션:** item이 '무선이어폰'이고 status가 '보관중'인 행의 모든 열을 조회하세요.

<details><summary>막힐 때 힌트</summary>

AND 양쪽 조건을 모두 만족해야 합니다.

</details>

In [ ]:
# TODO 07: 삼중따옴표 안을 완성하세요.
sql_07 = """
SELECT * FROM parcels
WHERE ___
  AND ___;
"""
run(sql_07)

## TODO 08 무게 단서까지 대조
**미션:** TODO 7의 조건에 weight_g 200 이상 300 이하를 추가하세요. 모든 열을 조회합니다.

<details><summary>막힐 때 힌트</summary>

200과 300을 모두 포함합니다. BETWEEN으로 써도 됩니다.

</details>

In [ ]:
# TODO 08: 삼중따옴표 안을 완성하세요.
sql_08 = """
SELECT * FROM parcels
WHERE ___
  AND ___
  AND ___
  AND ___;
"""
run(sql_08)

## TODO 09 가벼운 순으로 정리
**미션:** 전체 택배의 parcel_id, weight_g를 이 순서로 조회하세요. weight_g 오름차순, 동점이면 parcel_id 오름차순입니다.

<details><summary>막힐 때 힌트</summary>

ORDER BY는 WHERE 다음에 씁니다. 이 문제에는 WHERE가 필요 없습니다.

</details>

In [ ]:
# TODO 09: 삼중따옴표 안을 완성하세요.
sql_09 = """
SELECT ___, ___ FROM parcels
ORDER BY ___ ASC, ___ ASC;
"""
run(sql_09)

## TODO 10 확인할 택배 한 건 찾기
**미션:** TODO 8의 조건에 zone이 'B'라는 단서를 추가하세요. parcel_id, locker만 이 순서로 조회하세요.

<details><summary>막힐 때 힌트</summary>

어떤 사람의 잘못도 확정할 수 없습니다. 확인할 택배 후보를 찾는 문제입니다.

</details>

In [ ]:
# TODO 10: 삼중따옴표 안을 완성하세요.
sql_10 = """
SELECT ___, ___ FROM parcels
WHERE ___
  AND ___
  AND ___
  AND ___
  AND ___;
"""
run(sql_10)

## 마지막 자동 채점
각 문제 10점, 총 100점입니다. SQL 문자열 자체가 아니라 **결과의 열 개수·열 순서·값·중복**을 비교합니다. 9번은 행 순서도 확인합니다.

기본 기록과 번호·순서를 바꾸고 경계값을 추가한 기록에서 모두 맞아야 통과합니다. 다른 올바른 SQL도 허용합니다.

노트북 안 채점기는 자기 점검용으로, 수정이 가능하므로 부정행위를 막는 시험용 보안 채점기가 아닙니다.

In [ ]:
# 자기 점검용 채점기입니다. 수정하지 않고 실행하세요.
_REFERENCE = ['SELECT * FROM parcels;', 'SELECT parcel_id, item, locker FROM parcels;', "SELECT * FROM parcels WHERE item = '무선이어폰';", "SELECT * FROM parcels WHERE status = '수령완료';", 'SELECT * FROM parcels WHERE weight_g >= 200;', 'SELECT * FROM parcels WHERE weight_g <= 300;', "SELECT * FROM parcels WHERE item = '무선이어폰' AND status = '보관중';", "SELECT * FROM parcels WHERE item = '무선이어폰' AND status = '보관중' AND weight_g >= 200 AND weight_g <= 300;", 'SELECT parcel_id, weight_g FROM parcels ORDER BY weight_g ASC, parcel_id ASC;', "SELECT parcel_id, locker FROM parcels WHERE item = '무선이어폰' AND status = '보관중' AND weight_g >= 200 AND weight_g <= 300 AND zone = 'B';"]

def _variants():
    # 데이터가 달라져도 조건에 맞는 행을 찾는지 추가로 점검합니다.
    variant = []
    for i, row in enumerate(ROWS):
        p,item,w,z,loc,t,st = row
        variant.append((f"Q{9000+i}",item,w,z,loc,t,st))
    variant.extend([
        ("Q9901","무선이어폰",200,"B","B20","18:00","보관중"),
        ("Q9902","무선이어폰",300,"B","B21","18:01","보관중"),
        ("Q9903","무선이어폰",199,"B","B22","18:02","보관중"),
        ("Q9904","무선이어폰",301,"B","B23","18:03","보관중"),
        ("Q9905","보조배터리",240,"B","B24","18:04","보관중"),
        ("Q9906","무선이어폰",240,"B","B25","18:05","수령완료"),
        ("Q9907","무선이어폰",240,"A","A20","18:06","보관중")])
    return [None, list(reversed(variant))]

def check_one(number, sql):
    if not isinstance(sql,str) or not sql.strip() or 'TODO' in sql or '___' in sql:
        return False, "미작성: TODO를 SQL로 바꾸고 해당 셀을 실행하세요."
    for k,data in enumerate(_variants()):
        try:
            names, got = read_query(sql, data)
            ref_names, expected = read_query(_REFERENCE[number-1], data)
        except (sqlite3.Error, ValueError) as e:
            return False, "실행 오류: "+str(e)
        if len(names) != len(ref_names):
            return False, "조회할 열의 개수와 순서를 확인하세요."
        same = got == expected if number == 9 else Counter(got) == Counter(expected)
        if not same:
            if k:
                return False, "추가 데이터 점검: 고정 번호, 무게 경계 또는 빠진 조건을 확인하세요."
            return False, "결과 불일치: 열 순서, 조건, 중복 행" + (", 정렬" if number==9 else "") + "을 확인하세요."
    return True, "통과: 기본 기록과 추가 기록에서 확인했습니다."

def grade_all():
    global grading_result, score
    grading_result=[]
    for number in range(1,11):
        sql=globals().get(f"sql_{number:02d}","")
        ok,message=check_one(number,sql)
        grading_result.append({"mission":number,"points":10 if ok else 0,"feedback":message,"sql":sql})
        print(f"TODO {number:02d}  {'✓' if ok else '△'}  {10 if ok else 0}/10  {message}")
    score=sum(r['points'] for r in grading_result)
    print(f"\n총점 {score}/100점. 수정한 TODO 셀을 실행한 뒤 이 채점 셀을 다시 실행하세요.")
    print("자동 점수는 SQL 결과에 대한 자기 점검입니다. 근거 설명은 활동지에서 선생님과 확인합니다.")
    return score

grade_all()


## 수사 보고서
활동지에 후보 택배 번호와 보관함을 적고, 왜 다른 후보를 제외했는지 두 가지 이상 설명하세요.

조회 결과만으로 물건의 실제 내용, 소유자, 고의적인 바꿔치기를 확정할 수 있을까요? 다음 행동으로 “관리자에게 주문 내역과 보관함 기록 확인 요청”을 제안할 수 있습니다.

## 제출용 채점 결과 저장
점검 결과와 작성한 SQL을 CSV로 저장합니다. 코랩에서는 다운로드가 시작됩니다.

In [ ]:
# 제출용 CSV 만들기. 수정 후에는 채점 셀부터 다시 실행하세요.
grade_all()
output = io.StringIO(newline="")
writer = csv.DictWriter(output, fieldnames=["mission","points","feedback","sql"])
writer.writeheader()
writer.writerows(grading_result)
filename = "SQL탐정단_1차시_채점결과.csv"
with open(filename,"w",encoding="utf-8-sig",newline="") as f:
    f.write(output.getvalue())
print("저장:",filename,"/ 총점:",score)
try:
    from google.colab import files
    files.download(filename)
except ImportError:
    print("코랩이 아니면 현재 폴더에서 파일을 확인하세요.")


## 참고
- [SQLite SELECT 공식 문서](https://www.sqlite.org/lang_select.html)
- [CS50 SQL 교육과정](https://cs50.harvard.edu/sql/weeks/)

학습 순서를 참고했으며 이야기·데이터·문제는 이 수업용으로 새로 구성했습니다. 첨부 ZIP의 로컬 웹서버 실행 방식을 코랩의 메모리 SQLite 방식으로 바꾸었습니다.